In [ ]:
%%writefile submission.py
"""
Orbit Wars - Merged Elite Agent
Combines best strategies from:
  - Tactical Heuristic: sun waypoints, iterative aim, doomed evacuation, forward funnel
  - Elite Macro Physics: adaptive margins, synchronized swarm, tempo plays
  - Forward Sim + Snipe: mission scoring, snipe missions, retaliation discount, endgame
"""

import math
from collections import defaultdict
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

# ======================================================================
# CONSTANTS (mirror environment)
# ======================================================================
BOARD = 100.0
CX, CY = 50.0, 50.0
SUN_R = 10.0
ROT_LIMIT = 50.0
MAX_SPEED = 6.0
SPEED_REF = 1000.0
TOTAL_STEPS = 500
SUN_MARGIN = 1.3
PLANET_HIT = 1.0
SIM_HORIZON = 80
ENDGAME_TURN = 460
SYNC_WINDOW_SHORT = 2
SYNC_WINDOW_LONG = 4

# ======================================================================
# PERSISTENT STATE
# ======================================================================
_S = {"step": 0, "init_pos": {}, "omega": 0.0}


# ======================================================================
# GEOMETRY
# ======================================================================
def _dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def _hypot2(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def _pt_seg_dist(p, v, w):
    l2 = (v[0] - w[0]) ** 2 + (v[1] - w[1]) ** 2
    if l2 == 0.0:
        return _hypot2(p, v)
    t = max(0.0, min(1.0, ((p[0] - v[0]) * (w[0] - v[0]) + (p[1] - v[1]) * (w[1] - v[1])) / l2))
    return math.hypot(p[0] - (v[0] + t * (w[0] - v[0])), p[1] - (v[1] + t * (w[1] - v[1])))


def _crosses_sun(x1, y1, x2, y2, margin=SUN_MARGIN):
    return _pt_seg_dist((CX, CY), (x1, y1), (x2, y2)) < SUN_R + margin


def _fleet_speed(ships):
    if ships <= 1:
        return 1.0
    r = math.log(max(1, min(ships, 1000))) / math.log(SPEED_REF)
    return min(MAX_SPEED, 1.0 + (MAX_SPEED - 1.0) * max(0.0, r) ** 1.5)


def _is_orbiting(p):
    return _dist(p.x, p.y, CX, CY) + p.radius < ROT_LIMIT


# ======================================================================
# SUN-SAFE ANGLE (from Tactical — waypoint search)
# ======================================================================
def _safe_angle(sx, sy, tx, ty):
    """Returns (angle, estimated_distance). Uses waypoints to avoid sun."""
    if not _crosses_sun(sx, sy, tx, ty):
        return math.atan2(ty - sy, tx - sx), _dist(sx, sy, tx, ty)
    vx, vy = tx - sx, ty - sy
    norm = math.hypot(vx, vy)
    if norm < 1e-9:
        return math.atan2(ty - sy, tx - sx), norm
    nx, ny = -vy / norm, vx / norm
    best = None
    for sign in (1.0, -1.0):
        for mult in (1.8, 2.3, 3.0, 4.0):
            wx = CX + sign * nx * (SUN_R * mult)
            wy = CY + sign * ny * (SUN_R * mult)
            if _crosses_sun(sx, sy, wx, wy, margin=SUN_MARGIN) or \
               _crosses_sun(wx, wy, tx, ty, margin=SUN_MARGIN):
                continue
            d = _dist(sx, sy, wx, wy) + _dist(wx, wy, tx, ty)
            if best is None or d < best[0]:
                best = (d, wx, wy)
            break
    if best is None:
        return math.atan2(ty - sy, tx - sx), _dist(sx, sy, tx, ty) * 1.8
    _, wx, wy = best
    return math.atan2(wy - sy, wx - sx), best[0]


# ======================================================================
# POSITION PREDICTION
# ======================================================================
def _predict_orbit(init_xy, delta, omega, cur_step):
    ix, iy = init_xy
    dx, dy = ix - CX, iy - CY
    r = math.hypot(dx, dy)
    if r < 1e-6:
        return (ix, iy)
    th = math.atan2(dy, dx) + omega * (cur_step + delta)
    return (CX + r * math.cos(th), CY + r * math.sin(th))


def _predict_comet(path, idx, delta):
    fi = int(round(idx + delta))
    if fi < 0 or fi >= len(path):
        return None
    return (float(path[fi][0]), float(path[fi][1]))


def _predict_pos(planet, delta, ctx):
    cdata = ctx["comet_data"].get(planet.id)
    if cdata is not None:
        path, idx = cdata
        return _predict_comet(path, idx, delta)
    if not _is_orbiting(planet):
        return (planet.x, planet.y)
    init = _S["init_pos"].get(planet.id)
    if init is None:
        return (planet.x, planet.y)
    return _predict_orbit((init[0], init[1]), delta, ctx["omega"], ctx["step"])


def _intercept(src_xy, target, ships, ctx, max_iter=5, tol=0.04):
    """Iterative convergence intercept (from Tactical)."""
    v = _fleet_speed(ships)
    tx, ty = target.x, target.y
    t = _hypot2(src_xy, (tx, ty)) / max(v, 0.01)
    for _ in range(max_iter):
        pred = _predict_pos(target, t, ctx)
        if pred is None:
            return None, None, v
        tx, ty = pred
        _, d = _safe_angle(src_xy[0], src_xy[1], tx, ty)
        nt = d / max(v, 0.01)
        if abs(nt - t) < tol:
            t = nt
            break
        t = nt
    return t, (tx, ty), v


def _comet_life(planet_id, ctx):
    cdata = ctx["comet_data"].get(planet_id)
    if cdata is None:
        return 999
    path, idx = cdata
    return max(0, len(path) - idx)


# ======================================================================
# FLEET DESTINATION PROJECTION
# ======================================================================
def _fleet_dest(fleet, planets):
    dx, dy = math.cos(fleet.angle), math.sin(fleet.angle)
    speed = _fleet_speed(fleet.ships)
    best_pid, best_t = None, float("inf")
    for p in planets:
        if p.id == fleet.from_planet_id:
            continue
        tp = (p.x - fleet.x) * dx + (p.y - fleet.y) * dy
        if tp < 0:
            continue
        cx = fleet.x + tp * dx
        cy = fleet.y + tp * dy
        if math.hypot(cx - p.x, cy - p.y) < p.radius + PLANET_HIT:
            tt = tp / speed
            if tt < best_t:
                best_t = tt
                best_pid = p.id
    if best_pid is None:
        return None
    return best_pid, best_t


def _build_ledger(fleets, planets):
    ledger = defaultdict(list)
    for f in fleets:
        dest = _fleet_dest(f, planets)
        if dest is None:
            continue
        pid, tt = dest
        if tt <= TOTAL_STEPS:
            ledger[pid].append((tt, f.owner, f.ships))
    for pid in ledger:
        ledger[pid].sort()
    return ledger


# ======================================================================
# FORWARD SIMULATION (from Snipe — per-planet timeline)
# ======================================================================
def _simulate_planet(planet, arrivals, player, horizon=SIM_HORIZON):
    """Returns (owner_at, ships_at, deficit, first_enemy_arr)."""
    owner = planet.owner
    ships = float(planet.ships)
    prod = planet.production

    arr_by_turn = defaultdict(list)
    for t, o, s in arrivals:
        arr_by_turn[max(1, int(math.ceil(t)))].append((o, s))

    owner_at = {0: owner}
    ships_at = {0: ships}
    deficit = 0
    first_enemy = None

    for T in range(1, horizon + 1):
        if owner != -1:
            ships += prod
        if T in arr_by_turn:
            if planet.owner == player and first_enemy is None:
                for o, _ in arr_by_turn[T]:
                    if o != player and o != -1:
                        first_enemy = T
                        break
            by_owner = defaultdict(int)
            for o, s in arr_by_turn[T]:
                by_owner[o] += s
            so = sorted(by_owner.items(), key=lambda x: -x[1])
            if len(so) >= 2 and so[0][1] == so[1][1]:
                surv_o, surv_s = -1, 0
            elif len(so) >= 2:
                surv_o, surv_s = so[0][0], so[0][1] - so[1][1]
            else:
                surv_o, surv_s = so[0][0], so[0][1]
            if surv_s > 0:
                if surv_o == owner:
                    ships += surv_s
                else:
                    ships -= surv_s
                    if owner == player and ships < 0:
                        deficit = max(deficit, int(-ships + 1))
                    if ships < 0:
                        owner = surv_o
                        ships = -ships
        owner_at[T] = owner
        ships_at[T] = ships

    return owner_at, ships_at, deficit, first_enemy


# ======================================================================
# CONTEXT BUILDER
# ======================================================================
def _obs_get(obs, key, default=None):
    return obs.get(key, default) if isinstance(obs, dict) else getattr(obs, key, default)


def _build_context(obs):
    player = _obs_get(obs, "player", 0)
    raw_planets = _obs_get(obs, "planets", []) or []
    raw_fleets = _obs_get(obs, "fleets", []) or []
    omega = float(_obs_get(obs, "angular_velocity", 0.0) or 0.0)
    raw_initial = _obs_get(obs, "initial_planets", []) or []
    raw_comets = _obs_get(obs, "comets", []) or []
    comet_ids = set(_obs_get(obs, "comet_planet_ids", []) or [])
    obs_step = _obs_get(obs, "step", None)

    planets = [Planet(*p) for p in raw_planets]
    fleets = [Fleet(*f) for f in raw_fleets]
    pid2p = {p.id: p for p in planets}

    if not _S["init_pos"] and raw_initial:
        for ip in raw_initial:
            _S["init_pos"][ip[0]] = (ip[2], ip[3], ip[4])
    _S["omega"] = omega
    cur_step = int(obs_step) if obs_step is not None else _S["step"]

    comet_data = {}
    for group in raw_comets:
        if not isinstance(group, dict):
            continue
        pids = group.get("planet_ids") or []
        paths = group.get("paths") or []
        idx = int(group.get("path_index", 0) or 0)
        for pid, path in zip(pids, paths):
            comet_data[pid] = (path, idx)

    my_planets = [p for p in planets if p.owner == player]
    enemies = [p for p in planets if p.owner != player and p.owner != -1]

    enemy_dist = {}
    for p in planets:
        if not enemies:
            enemy_dist[p.id] = 200.0
        else:
            enemy_dist[p.id] = min(_hypot2((p.x, p.y), (e.x, e.y)) for e in enemies)

    ledger = _build_ledger(fleets, planets)

    owner_at, ships_at, deficit_map, first_enemy = {}, {}, {}, {}
    for p in planets:
        oa, sa, d, fe = _simulate_planet(p, ledger.get(p.id, []), player)
        owner_at[p.id] = oa
        ships_at[p.id] = sa
        deficit_map[p.id] = d
        first_enemy[p.id] = fe

    # Adaptive aggression (from Elite)
    my_total = sum(p.ships for p in my_planets) + sum(f.ships for f in fleets if f.owner == player)
    enemy_total = sum(p.ships for p in enemies) + \
                  sum(f.ships for f in fleets if f.owner != player and f.owner != -1)
    my_prod = sum(p.production for p in my_planets)
    enemy_prod = sum(p.production for p in enemies)
    domination = (my_total - enemy_total) / max(1, my_total + enemy_total)

    attack_margin = 1.12
    if my_total > enemy_total * 1.3:
        attack_margin += 0.10
    elif my_total < enemy_total * 0.8:
        attack_margin -= 0.05

    remaining = max(1, TOTAL_STEPS - cur_step)

    # Indirect wealth map (from Tactical)
    indirect_w = {}
    for p in planets:
        w = 0.0
        for q in planets:
            if q.id == p.id:
                continue
            d = _dist(p.x, p.y, q.x, q.y)
            if d < 1:
                continue
            factor = q.production / (d + 10.0)
            if q.owner == player:
                w += factor * 0.5
            elif q.owner == -1:
                w += factor * 1.0
            else:
                w += factor * 1.3
        indirect_w[p.id] = w

    return {
        "player": player, "planets": planets, "fleets": fleets, "pid2p": pid2p,
        "my_planets": my_planets, "enemies": enemies,
        "omega": omega, "step": cur_step, "remaining": remaining,
        "comet_ids": comet_ids, "comet_data": comet_data,
        "enemy_dist": enemy_dist, "ledger": ledger,
        "owner_at": owner_at, "ships_at": ships_at,
        "deficit": deficit_map, "first_enemy": first_enemy,
        "my_total": my_total, "enemy_total": enemy_total,
        "my_prod": my_prod, "enemy_prod": enemy_prod,
        "domination": domination, "attack_margin": attack_margin,
        "indirect_w": indirect_w,
        "is_early": cur_step < 35, "is_late": remaining < 60,
        "is_very_late": remaining < 25,
        "is_endgame": cur_step >= ENDGAME_TURN,
        "finishing": domination > 0.35 and my_prod > enemy_prod * 1.3 and cur_step > 100,
        "behind": domination < -0.25,
    }


def _state_at(ctx, target_id, arrival_turn):
    T = min(SIM_HORIZON, max(0, int(math.ceil(arrival_turn))))
    return (ctx["owner_at"][target_id].get(T, ctx["owner_at"][target_id][0]),
            max(0.0, ctx["ships_at"][target_id].get(T, ctx["ships_at"][target_id][0])))


# ======================================================================
# SCORING (from Snipe — sigmoid retaliation + indirect wealth from Tactical)
# ======================================================================
def _value(target, arrival_turn, ctx, is_reinforce=False):
    remaining = max(1.0, ctx["remaining"] - arrival_turn)
    if target.id in ctx["comet_ids"]:
        life = _comet_life(target.id, ctx)
        remaining = min(remaining, max(0, life - arrival_turn))
        if remaining <= 0:
            return 0.0
    base = target.production * remaining
    # indirect wealth bonus (from Tactical)
    iw = ctx["indirect_w"].get(target.id, 0.0)
    base += iw * remaining * 0.12
    # sigmoid retaliation discount (from Snipe)
    d = ctx["enemy_dist"].get(target.id, 100.0)
    discount = 1.0 / (1.0 + math.exp(-(d - 22.0) / 7.0))
    if is_reinforce:
        discount *= 0.9
    return base * discount


def _launchable(mine, ctx):
    """Safe ships to send, keeping defense reserve (from Snipe + Tactical)."""
    danger = 1.0 / (1.0 + math.exp((ctx["enemy_dist"].get(mine.id, 100.0) - 20.0) / 6.0))
    min_keep = int(max(2, mine.ships * (0.05 + 0.25 * danger)))
    deficit = ctx["deficit"].get(mine.id, 0)
    return max(0, mine.ships - min_keep - deficit)


# ======================================================================
# MISSION BUILDERS
# ======================================================================
def _build_capture(mine, target, ctx):
    if target.owner == ctx["player"] or target.id == mine.id:
        return None
    probe = max(5, min(int(mine.ships * 0.7), int(target.ships) + 20))
    t, pos, _ = _intercept((mine.x, mine.y), target, probe, ctx)
    if t is None or pos is None or t > ctx["remaining"]:
        return None
    if _crosses_sun(mine.x, mine.y, pos[0], pos[1]):
        return None

    owner_arr, ships_arr = _state_at(ctx, target.id, t)
    if owner_arr == ctx["player"]:
        return None

    ships_needed = int(math.ceil(ships_arr * ctx["attack_margin"])) + 1
    launchable = _launchable(mine, ctx)
    if ships_needed > launchable or ships_needed < 1:
        return None

    t_f, pos_f, _ = _intercept((mine.x, mine.y), target, ships_needed, ctx)
    if t_f is None or pos_f is None:
        return None
    if _crosses_sun(mine.x, mine.y, pos_f[0], pos_f[1]):
        return None
    owner_f, ships_f = _state_at(ctx, target.id, t_f)
    if owner_f == ctx["player"]:
        return None
    ships_needed = max(ships_needed, int(math.ceil(ships_f * ctx["attack_margin"])) + 1)
    if ships_needed > launchable:
        return None

    if target.id in ctx["comet_ids"] and t_f >= _comet_life(target.id, ctx):
        return None

    mtype = "expand" if target.owner == -1 else "attack"
    value = _value(target, t_f, ctx)
    # Early expansion bonus (from Tactical)
    if ctx["is_early"] and mtype == "expand" and ships_needed <= 18:
        value *= 1.5
    # Attack bonus when finishing (from Tactical)
    if mtype == "attack" and ctx["finishing"]:
        value *= 1.4
    angle, _ = _safe_angle(mine.x, mine.y, pos_f[0], pos_f[1])
    return {
        "type": mtype, "target_id": target.id,
        "sources": [(mine.id, ships_needed, angle)],
        "arrival_turn": t_f, "cost": ships_needed, "value": value,
    }


def _build_snipe(mine, target, enemy_arr, ctx):
    """Snipe: arrive at neutral same turn as enemy fleet (from Snipe agent)."""
    if target.owner != -1 or target.id == mine.id:
        return None
    probe = max(10, int(target.ships) + 40)
    t, pos, _ = _intercept((mine.x, mine.y), target, probe, ctx)
    if t is None or pos is None:
        return None
    if t > enemy_arr + 1.5 or t < enemy_arr - 1.5:
        return None
    if _crosses_sun(mine.x, mine.y, pos[0], pos[1]):
        return None

    sync = max(t, enemy_arr)
    owner_s, ships_s = _state_at(ctx, target.id, sync)
    if owner_s == ctx["player"]:
        return None

    need = int(math.ceil(ships_s * ctx["attack_margin"])) + 1
    launchable = _launchable(mine, ctx)
    if need > launchable or need < 1:
        return None

    t_f, pos_f, _ = _intercept((mine.x, mine.y), target, need, ctx)
    if t_f is None or pos_f is None:
        return None
    if _crosses_sun(mine.x, mine.y, pos_f[0], pos_f[1]):
        return None
    angle, _ = _safe_angle(mine.x, mine.y, pos_f[0], pos_f[1])
    value = _value(target, t_f, ctx)
    return {
        "type": "snipe", "target_id": target.id,
        "sources": [(mine.id, need, angle)],
        "arrival_turn": t_f, "cost": need, "value": value,
    }


def _build_reinforce(ally, threatened, ctx):
    if ally.id == threatened.id:
        return None
    deficit = ctx["deficit"].get(threatened.id, 0)
    if deficit <= 0:
        return None
    crit = ctx["first_enemy"].get(threatened.id)
    if crit is None or crit <= 0:
        return None

    probe = max(5, deficit + 3)
    t, pos, _ = _intercept((ally.x, ally.y), threatened, probe, ctx)
    if t is None or pos is None or t >= crit:
        return None
    if _crosses_sun(ally.x, ally.y, pos[0], pos[1]):
        return None

    ships = min(max(deficit + 2, 3), max(1, ally.ships - 2))
    if ships < 1:
        return None
    angle, _ = _safe_angle(ally.x, ally.y, pos[0], pos[1])
    value = _value(threatened, t, ctx, is_reinforce=True)
    return {
        "type": "reinforce", "target_id": threatened.id,
        "sources": [(ally.id, ships, angle)],
        "arrival_turn": t, "cost": ships, "value": value,
    }


# ======================================================================
# SYNCHRONIZED SWARM (from Elite — sliding window)
# ======================================================================
def _build_swarm(target, front_planets, ctx):
    """Try to find a synchronized swarm attack within a time window."""
    if target.owner == ctx["player"]:
        return None
    potential = []
    for f in front_planets:
        avail = _launchable(f, ctx)
        if avail <= 0:
            continue
        t, pos, _ = _intercept((f.x, f.y), target, avail, ctx)
        if t is None or pos is None:
            continue
        if _crosses_sun(f.x, f.y, pos[0], pos[1]):
            continue
        angle, _ = _safe_angle(f.x, f.y, pos[0], pos[1])
        potential.append({"id": f.id, "t_a": t, "angle": angle, "ships": avail})

    if not potential:
        return None
    potential.sort(key=lambda x: x["t_a"])

    # Find best sync window
    best_window, best_ships, best_t = [], 0, potential[0]["t_a"]
    sync_w = SYNC_WINDOW_SHORT if best_t < 15 else (SYNC_WINDOW_LONG if best_t > 40 else 3)
    for anchor in potential:
        window = [p for p in potential if anchor["t_a"] <= p["t_a"] <= anchor["t_a"] + sync_w]
        gathered = sum(p["ships"] for p in window)
        if gathered > best_ships:
            best_ships, best_window, best_t = gathered, window, anchor["t_a"]

    owner_arr, ships_arr = _state_at(ctx, target.id, best_t)
    if owner_arr == ctx["player"]:
        return None
    needed = int(math.ceil(ships_arr * ctx["attack_margin"])) + 1

    if best_ships >= needed:
        # Full strike
        sources = []
        gathered = 0
        for p in sorted(best_window, key=lambda x: x["t_a"]):
            amt = min(p["ships"], needed - gathered)
            if amt > 0:
                sources.append((p["id"], amt, p["angle"]))
                gathered += amt
            if gathered >= needed:
                break
        value = _value(ctx["pid2p"][target.id], best_t, ctx)
        if ctx["finishing"]:
            value *= 1.3
        return {
            "type": "swarm", "target_id": target.id,
            "sources": sources,
            "arrival_turn": best_t, "cost": gathered, "value": value,
        }

    # Tempo play: 70% threshold partial attack (from Elite)
    if best_ships >= 0.7 * needed and not ctx["is_late"]:
        sources = []
        for p in best_window:
            amt = int(p["ships"] * 0.85)
            if amt > 0:
                sources.append((p["id"], amt, p["angle"]))
        total_cost = sum(s for _, s, _ in sources)
        value = _value(ctx["pid2p"][target.id], best_t, ctx) * 0.6
        return {
            "type": "tempo", "target_id": target.id,
            "sources": sources,
            "arrival_turn": best_t, "cost": total_cost, "value": value,
        }

    return None


# ======================================================================
# DOOMED EVACUATION (from Tactical)
# ======================================================================
def _evacuation_moves(ctx, allocated):
    moves = []
    player = ctx["player"]
    my_planets = ctx["my_planets"]
    doomed = set()

    for p in my_planets:
        oa = ctx["owner_at"].get(p.id, {})
        for T in range(1, SIM_HORIZON + 1):
            if oa.get(T, p.owner) != player and oa.get(T, p.owner) != -1:
                if ctx["deficit"].get(p.id, 0) >= p.ships:
                    doomed.add(p.id)
                break

    safe_planets = [p for p in my_planets if p.id not in doomed and p.id not in allocated]
    for pid in doomed:
        p = ctx["pid2p"][pid]
        avail = p.ships - allocated.get(pid, 0)
        if avail <= 0 or not safe_planets:
            continue
        # Attack nearest enemy if possible, else retreat to nearest ally
        best_tgt = None
        best_score = -1
        for e in ctx["enemies"]:
            d = _dist(p.x, p.y, e.x, e.y)
            if d > 100:
                continue
            t_est = d / max(_fleet_speed(avail), 0.01)
            if t_est >= ctx["remaining"] - 2:
                continue
            score = e.production * max(1, ctx["remaining"] - t_est) * 0.8 / (t_est + 1)
            if score > best_score:
                best_score = score
                best_tgt = e

        if best_tgt is None:
            best_tgt = min(safe_planets, key=lambda q: _dist(p.x, p.y, q.x, q.y))

        angle, _ = _safe_angle(p.x, p.y, best_tgt.x, best_tgt.y)
        if not _crosses_sun(p.x, p.y, p.x + math.cos(angle) * 3, p.y + math.sin(angle) * 3, margin=0.4):
            moves.append([pid, float(angle), int(avail)])
            allocated[pid] = allocated.get(pid, 0) + avail

    return moves, doomed


# ======================================================================
# FORWARD FUNNEL (from Tactical — rear sends to front)
# ======================================================================
def _funnel_moves(ctx, allocated, doomed):
    moves = []
    player = ctx["player"]
    my_planets = ctx["my_planets"]
    enemies = ctx["enemies"]
    neutrals = [p for p in ctx["planets"] if p.owner == -1]

    if not my_planets or len(my_planets) < 2 or ctx["is_very_late"]:
        return moves
    if not enemies and not neutrals:
        return moves

    ref_set = enemies if enemies else neutrals
    front_dist = {p.id: min((_dist(p.x, p.y, e.x, e.y) for e in ref_set), default=200) for p in my_planets}
    front = min(my_planets, key=lambda p: front_dist[p.id])

    send_ratio = 0.8 if ctx["finishing"] else 0.6

    for r in sorted(my_planets, key=lambda p: -front_dist[p.id]):
        if r.id == front.id or r.id in doomed:
            continue
        if front_dist[r.id] < front_dist[front.id] * 1.2:
            continue
        avail = max(0, r.ships - allocated.get(r.id, 0))
        if avail < 15:
            continue
        # Find a closer ally to funnel through
        mid = [p for p in my_planets if p.id != r.id and p.id not in doomed
               and front_dist[p.id] < front_dist[r.id] * 0.75]
        fwd = mid[0] if mid else front
        if fwd.id == r.id:
            continue
        send = int(avail * send_ratio)
        if send < 10:
            continue
        t, pos, _ = _intercept((r.x, r.y), fwd, send, ctx)
        if t is None or pos is None or t > 40:
            continue
        angle, _ = _safe_angle(r.x, r.y, pos[0], pos[1])
        if _crosses_sun(r.x, r.y, r.x + math.cos(angle) * 3, r.y + math.sin(angle) * 3, margin=0.4):
            continue
        moves.append([r.id, float(angle), int(send)])
        allocated[r.id] = allocated.get(r.id, 0) + send

    return moves


# ======================================================================
# MISSION SCORING
# ======================================================================
def _score(m, ctx):
    s = m["value"] / (m["cost"] + 5.0)
    s -= 0.015 * m["arrival_turn"]
    if m["type"] == "snipe":
        s *= 1.4
    elif m["type"] == "reinforce":
        s *= 2.0
    elif m["type"] == "swarm":
        s *= 1.2
    elif m["type"] == "tempo":
        s *= 0.8
    if m["target_id"] in ctx["comet_ids"]:
        s *= 1.15
    if ctx["is_endgame"] and m["type"] not in ("reinforce", "swarm"):
        s *= 0.25
    return s


# ======================================================================
# ALLOCATION
# ======================================================================
def _allocate(missions, ctx, allocated):
    missions.sort(key=lambda m: -m["score"])
    targeted = set()
    moves = []
    pid2p = ctx["pid2p"]
    player = ctx["player"]
    for m in missions:
        if m["score"] <= 0.01:
            break
        if m["target_id"] in targeted:
            continue
        ok = True
        for src_id, need, _ in m["sources"]:
            src = pid2p.get(src_id)
            if src is None or src.owner != player:
                ok = False
                break
            free = src.ships - allocated.get(src_id, 0)
            if free < need:
                ok = False
                break
        if not ok:
            continue
        for src_id, need, angle in m["sources"]:
            moves.append([src_id, float(angle), int(need)])
            allocated[src_id] = allocated.get(src_id, 0) + need
        targeted.add(m["target_id"])
    return moves


# ======================================================================
# MAIN AGENT
# ======================================================================
def agent(obs, config=None):
    ctx = _build_context(obs)
    player = ctx["player"]
    planets = ctx["planets"]
    my_planets = ctx["my_planets"]

    if not my_planets:
        _S["step"] = ctx["step"] + 1
        return []

    allocated = {}
    all_moves = []

    # Phase 1: Doomed evacuation (from Tactical)
    evac_moves, doomed = _evacuation_moves(ctx, allocated)
    all_moves.extend(evac_moves)

    # Phase 2: Build missions
    missions = []

    # Capture & attack missions
    for mine in my_planets:
        if mine.id in doomed:
            continue
        for target in planets:
            m = _build_capture(mine, target, ctx)
            if m is not None:
                missions.append(m)

    # Snipe missions (from Snipe agent)
    for target in planets:
        if target.owner != -1:
            continue
        enemy_arrs = [t for t, o, _ in ctx["ledger"].get(target.id, [])
                      if o != player and o != -1]
        if not enemy_arrs:
            continue
        ea = min(enemy_arrs)
        for mine in my_planets:
            if mine.id in doomed:
                continue
            m = _build_snipe(mine, target, ea, ctx)
            if m is not None:
                missions.append(m)

    # Reinforce missions
    threatened = [p for p in my_planets if ctx["deficit"].get(p.id, 0) > 0 and p.id not in doomed]
    safe = [p for p in my_planets if ctx["deficit"].get(p.id, 0) == 0 and p.id not in doomed]
    for t_planet in threatened:
        for ally in sorted(safe, key=lambda a: _hypot2((a.x, a.y), (t_planet.x, t_planet.y)))[:3]:
            m = _build_reinforce(ally, t_planet, ctx)
            if m is not None:
                missions.append(m)

    # Synchronized swarm (from Elite) — for enemy targets only
    front = [p for p in my_planets if p.id not in doomed
             and ctx["enemy_dist"].get(p.id, 200) < 50]
    if not front:
        front = [p for p in my_planets if p.id not in doomed]
    for target in planets:
        if target.owner == player or target.owner == -1:
            continue
        m = _build_swarm(target, front, ctx)
        if m is not None:
            missions.append(m)

    # Score all missions
    for m in missions:
        m["score"] = _score(m, ctx)

    # Allocate missions
    mission_moves = _allocate(missions, ctx, allocated)
    all_moves.extend(mission_moves)

    # Phase 3: Forward funnel (from Tactical)
    funnel_moves = _funnel_moves(ctx, allocated, doomed)
    all_moves.extend(funnel_moves)

    # Deduplicate & validate
    dedup = {}
    for sid, ang, sh in all_moves:
        key = (sid, round(ang, 4))
        if key in dedup:
            dedup[key] = (sid, ang, dedup[key][2] + sh)
        else:
            dedup[key] = (sid, ang, sh)

    final_moves = []
    used = {}
    for sid, ang, sh in dedup.values():
        src = ctx["pid2p"].get(sid)
        if src is None:
            continue
        max_ok = src.ships - used.get(sid, 0)
        send = min(sh, max_ok)
        if send >= 1:
            final_moves.append([sid, float(ang), int(send)])
            used[sid] = used.get(sid, 0) + send

    _S["step"] = ctx["step"] + 1
    return final_moves
